# Benchmark sain/fautif des capteurs ABS

CNN, GRU et LSTM sont chargés et entraînés dans des cellules indépendantes. Après chaque entraînement, le modèle retourne sur CPU et le cache CUDA inutilisé est libéré. Le test reste fermé jusqu'à la sélection validation.

## 1. Imports, reproductibilité et chemins

In [ ]:
from pathlib import Path
import gc
import random
import sys

import numpy as np
import pandas as pd
import torch

cwd = Path.cwd().resolve()
if cwd.name == 'notebooks':
    PROJECT_ROOT = cwd.parents[1]
elif cwd.name == 'fault_parameter_training':
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd
sys.path.insert(0, str(PROJECT_ROOT))

from fault_parameter_training.data import create_dataloaders, prepare_npz_dataset
from fault_parameter_training.models import (
    CNNFaultParameterEstimator,
    CNNGRUFaultParameterEstimator,
    GRUFaultParameterEstimator,
    LSTMFaultParameterEstimator,
)
from fault_parameter_training.monitoring import plot_benchmark, plot_histories
from fault_parameter_training.trainer import FaultParameterTrainer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

RESULTS = PROJECT_ROOT / 'ABS_SoH_Simulator' / 'simulation_results'
FAULTY_DATASET_CSV = sorted(RESULTS.glob('abs_faulty_braking_dataset_*.csv'))[-1]
FAULTY_MANIFEST_CSV = RESULTS / FAULTY_DATASET_CSV.name.replace('_dataset_', '_manifest_')
HEALTHY_DATASET_CSV = sorted(RESULTS.glob('abs_healthy_braking_dataset_*.csv'))[-1]
HEALTHY_MANIFEST_CSV = RESULTS / HEALTHY_DATASET_CSV.name.replace('_dataset_', '_manifest_')
CACHE = PROJECT_ROOT / 'fault_parameter_training' / 'cache' / 'fault_parameter_dataset.npz'
EXPERIMENTS = PROJECT_ROOT / 'fault_parameter_training' / 'experiments'
print('Python :', sys.executable)
print('Device :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Paramètres de l'expérience

In [ ]:
REBUILD_CACHE = False  # True si MAX_SIMULATIONS change
MAX_SIMULATIONS = None  # 64 pour un smoke test
BATCH_SIZE = 32  # augmenter uniquement si la mémoire GPU le permet
EPOCHS = 50
PATIENCE = 8
LEARNING_RATE = 1e-3
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

## 3. Dataset mixte

Chaque scénario fautif fournit une roue positive et trois roues saines. Chaque scénario sain ajoute une roue saine. Les paramètres sont masqués dans la loss pour les exemples sains.

In [ ]:
if REBUILD_CACHE or not CACHE.exists():
    prepare_npz_dataset(
        FAULTY_DATASET_CSV,
        FAULTY_MANIFEST_CSV,
        HEALTHY_DATASET_CSV,
        HEALTHY_MANIFEST_CSV,
        CACHE,
        max_simulations=MAX_SIMULATIONS,
        split_seed=SEED,
    )
else:
    print(f'Cache réutilisé : {CACHE}')

train_loader, validation_loader, test_loader = create_dataloaders(
    CACHE, batch_size=BATCH_SIZE, seed=SEED
)
pd.Series({
    'train': len(train_loader.dataset),
    'validation': len(validation_loader.dataset),
    'test': len(test_loader.dataset),
    'train_faulty': train_loader.dataset.positive_count,
    'train_healthy': len(train_loader.dataset) - train_loader.dataset.positive_count,
})

## 4. Trainer commun

In [ ]:
trainer = FaultParameterTrainer(
    train_loader,
    validation_loader,
    test_loader,
    output_directory=EXPERIMENTS,
    device=DEVICE,
)
print('Poids positif BCE :', trainer.positive_weight)

## 5. CNN — chargement et entraînement

Exécuter cette cellule seule pour entraîner ou réentraîner uniquement le CNN.

In [ ]:
cnn_model = CNNFaultParameterEstimator()
print('Paramètres CNN :', cnn_model.count_parameters())
cnn_history = trainer.train(
    cnn_model,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    patience=PATIENCE,
)
cnn_model.to('cpu')
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 6. GRU — chargement et entraînement

Exécuter cette cellule seule pour entraîner ou réentraîner uniquement la GRU.

In [ ]:
gru_model = GRUFaultParameterEstimator()
print('Paramètres GRU :', gru_model.count_parameters())
gru_history = trainer.train(
    gru_model,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    patience=PATIENCE,
)
gru_model.to('cpu')
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 7. CNN–GRU hybride — chargement et entraînement

Le CNN extrait les ruptures locales à une résolution de 40 ms, puis la GRU bidirectionnelle encode leur position et leur évolution temporelle.

In [ ]:
cnn_gru_model = CNNGRUFaultParameterEstimator()
print('Paramètres CNN–GRU :', cnn_gru_model.count_parameters())
cnn_gru_history = trainer.train(
    cnn_gru_model,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    patience=PATIENCE,
)
cnn_gru_model.to('cpu')
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 8. LSTM — chargement et entraînement

Exécuter cette cellule seule pour entraîner ou réentraîner uniquement la LSTM.

In [ ]:
lstm_model = LSTMFaultParameterEstimator()
print('Paramètres LSTM :', lstm_model.count_parameters())
lstm_history = trainer.train(
    lstm_model,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    patience=PATIENCE,
)
lstm_model.to('cpu')
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 9. Monitoring comparé train/validation

In [ ]:
models = {
    'cnn': cnn_model,
    'gru': gru_model,
    'cnn_gru': cnn_gru_model,
    'lstm': lstm_model,
}
histories = {
    'cnn': cnn_history,
    'gru': gru_history,
    'cnn_gru': cnn_gru_history,
    'lstm': lstm_history,
}
plot_histories(histories);

TensorBoard : `tensorboard --logdir fault_parameter_training/experiments`. Chaque modèle possède son propre historique et son propre checkpoint.

## 10. Sélection validation, puis évaluation test modèle par modèle

In [ ]:
validation_scores = {name: history['validation_loss'].min() for name, history in histories.items()}
winner_name = min(validation_scores, key=validation_scores.get)
print('Gagnant selon validation :', winner_name, validation_scores[winner_name])

benchmark_rows = []
test_predictions = {}
for name, model in models.items():
    metrics, test_predictions[name] = trainer.evaluate_test(model)
    benchmark_rows.append({
        'model': name,
        'parameters': model.count_parameters(),
        'validation_loss': validation_scores[name],
        **{f'test_{key}': value for key, value in metrics.items()},
    })
    model.to('cpu')
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

benchmark = pd.DataFrame(benchmark_rows).sort_values('validation_loss')
benchmark.to_csv(EXPERIMENTS / 'benchmark.csv', index=False)
benchmark

In [ ]:
plot_benchmark(benchmark);

## 11. Export ONNX du gagnant

In [ ]:
winner_onnx = trainer.export_onnx(models[winner_name])
print('Modèle exporté :', winner_onnx)
test_predictions[winner_name].head(10)

## 12. Inférence avec la fusion des CNN et GRU entraînés

La fusion moyenne les probabilités de défaut, utilise le CNN pour le début et la durée, et la GRU pour la sévérité. L'exemple reconstruit une série brute depuis le cache uniquement pour démontrer l'API.

In [ ]:
from fault_parameter_training.inference import CNNGRUFusionPredictor

fusion_predictor = CNNGRUFusionPredictor(
    EXPERIMENTS,
    CACHE,
    device=DEVICE,
)
sample = test_loader.dataset[0]
raw_speed_mps = (
    sample['x'][:, 0].numpy() * test_loader.dataset.speed_std
    + test_loader.dataset.speed_mean
)
valid = sample['x'][:, 1].numpy().astype(bool)
fusion_result = fusion_predictor.predict(raw_speed_mps, valid)
fusion_result